# Pipeline Setup (Dataset Generation)

This notebook is the **feature pipeline**: it consumes raw COCO val2017 images and produces the cached numerical artifacts that the trained model in `03_trained_scoring_head.ipynb` consumes. Run this **once per dataset configuration** (e.g., once per `NUM_IMAGES` setting). All outputs are written to Google Drive so subsequent notebooks can reuse them.

**What this notebook produces (saved to `/content/drive/MyDrive/model-cache/cache/`):**

| File | What's in it |
|---|---|
| `sample_manifest.pkl` | sampled image IDs, paths, COCO category memberships per image, all 80 COCO category names |
| `clip_embeddings.npy` | CLIP whole-image embeddings (one row per image) |
| `siglip_embeddings.npy` | SigLIP2 whole-image embeddings (one row per image) |
| `sam2_region_index.pkl` | SAM2 region embeddings + region-to-image map + bboxes + areas + IoU confidences |

**Runtime expectation (on Colab T4):**
- Image encoding: ~2 minutes
- SAM2 region indexing at NUM_IMAGES=500: ~45 minutes

**Why this is a separate notebook:** the trained model (B-2 deliverable) is tabular classification on the 9 features computed from these caches. Once the caches exist, `02b` runs end-to-end in under 5 minutes. Splitting lets you iterate on the model without re-running the slow pipeline.


## 1. Setup and Google Drive Mount


In [1]:
!pip install -q torch torchvision \
    "open_clip_torch>=2.31.0" "timm>=1.0.15" \
    "git+https://github.com/facebookresearch/sam2.git" \
    pycocotools pillow numpy


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import pickle
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO

DATA_DIR = Path("./coco_data"); DATA_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR = Path("./weights"); WEIGHTS_DIR.mkdir(exist_ok=True)

# Mount Drive and use it as the cache directory so trained-model notebooks pick up the artifacts
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/model-cache")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(exist_ok=True)
    print(f"Cache: {CACHE_DIR} (Drive-backed)")
except ImportError:
    CACHE_DIR = Path("./cache")
    CACHE_DIR.mkdir(exist_ok=True)
    print(f"Cache: {CACHE_DIR} (local)")

NUM_IMAGES = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Sample size target: {NUM_IMAGES} images")


Mounted at /content/drive
Cache: /content/drive/MyDrive/model-cache (Drive-backed)
Device: cuda
Sample size target: 500 images


## 2. Download COCO val2017 and SAM2 Weights


In [3]:
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMG_URL = "http://images.cocodataset.org/zips/val2017.zip"
SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt"

ANN_ZIP = DATA_DIR / "annotations_trainval2017.zip"
IMG_ZIP = DATA_DIR / "val2017.zip"
ANN_FILE = DATA_DIR / "annotations" / "instances_val2017.json"
IMG_DIR = DATA_DIR / "val2017"
SAM2_CKPT = WEIGHTS_DIR / "sam2.1_hiera_base_plus.pt"
SAM2_CFG = "configs/sam2.1/sam2.1_hiera_b+.yaml"

def download_if_missing(url, dest):
    if not dest.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, dest)

download_if_missing(ANN_URL, ANN_ZIP)
download_if_missing(IMG_URL, IMG_ZIP)
download_if_missing(SAM2_URL, SAM2_CKPT)

if not ANN_FILE.exists():
    with zipfile.ZipFile(ANN_ZIP, "r") as z: z.extractall(DATA_DIR)
if not IMG_DIR.exists():
    print("Extracting images (a few minutes)...")
    with zipfile.ZipFile(IMG_ZIP, "r") as z: z.extractall(DATA_DIR)
print("Assets ready.")


Extracting images (a few minutes)...
Assets ready.


## 3. Sample Images and Save Manifest

We pick a deterministic random sample of 500 images, record each image's COCO category memberships, and save a single manifest pickle that downstream notebooks will load instead of re-reading the COCO annotations file.


In [4]:
manifest_path = CACHE_DIR / "sample_manifest.pkl"

if manifest_path.exists():
    with open(manifest_path, "rb") as f:
        manifest = pickle.load(f)
    sampled_ids = manifest["sampled_ids"]
    sample_paths = manifest["sample_paths"]
    sample_categories = manifest["sample_categories"]
    COCO_CATEGORIES = manifest["coco_categories"]
    print(f"Loaded existing manifest: {len(sample_paths)} images")
else:
    coco = COCO(str(ANN_FILE))
    all_img_ids = coco.getImgIds()
    print(f"COCO val2017: {len(all_img_ids)} images, {len(coco.getCatIds())} categories")

    np.random.seed(42)
    sampled_ids = np.random.choice(all_img_ids, size=min(NUM_IMAGES, len(all_img_ids)), replace=False).tolist()

    sample_paths = []
    sample_categories = {}
    for iid in sampled_ids:
        img = coco.loadImgs(iid)[0]
        p = IMG_DIR / img["file_name"]
        if p.exists():
            sample_paths.append(str(p))
            anns = coco.loadAnns(coco.getAnnIds(imgIds=iid))
            sample_categories[iid] = {coco.loadCats(a["category_id"])[0]["name"] for a in anns}

    COCO_CATEGORIES = [c["name"] for c in coco.loadCats(coco.getCatIds())]

    manifest = {
        "sampled_ids": sampled_ids,
        "sample_paths": sample_paths,
        "sample_categories": sample_categories,
        "coco_categories": COCO_CATEGORIES,
        "num_images_target": NUM_IMAGES,
    }
    with open(manifest_path, "wb") as f:
        pickle.dump(manifest, f)
    print(f"Saved manifest: {len(sample_paths)} images, {len(COCO_CATEGORIES)} categories")

NUM_IMAGES = len(sample_paths)
print(f"Effective sample size: {NUM_IMAGES}")


loading annotations into memory...
Done (t=0.94s)
creating index...
index created!
COCO val2017: 5000 images, 80 categories
Saved manifest: 500 images, 80 categories
Effective sample size: 500


## 4. Load Encoders (CLIP + SigLIP2)


In [5]:
import open_clip

print("Loading CLIP ViT-B/32...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_model = clip_model.to(DEVICE).eval()

print("Loading SigLIP2 ViT-B/16-256...")
siglip_model, siglip_preprocess = open_clip.create_model_from_pretrained("hf-hub:timm/ViT-B-16-SigLIP2-256")
siglip_model = siglip_model.to(DEVICE).eval()
print("Encoders loaded.")


Loading CLIP ViT-B/32...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading SigLIP2 ViT-B/16-256...


open_clip_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Encoders loaded.


## 5. Encode All Images (Whole-Image)


In [6]:
def encode_images(model, preprocess, paths, batch_size=64):
    embeds = []
    for i in range(0, len(paths), batch_size):
        batch = [preprocess(Image.open(p).convert("RGB")) for p in paths[i:i+batch_size]]
        x = torch.stack(batch).to(DEVICE)
        with torch.no_grad():
            e = model.encode_image(x)
            e = e / e.norm(dim=-1, keepdim=True)
        embeds.append(e.cpu())
    return torch.cat(embeds).numpy()

clip_cache = CACHE_DIR / "clip_embeddings.npy"
if clip_cache.exists():
    clip_embeds = np.load(clip_cache)
    print(f"CLIP cache hit: {clip_embeds.shape}")
else:
    print("Encoding with CLIP...")
    clip_embeds = encode_images(clip_model, clip_preprocess, sample_paths)
    np.save(clip_cache, clip_embeds)
    print(f"Saved CLIP embeddings: {clip_embeds.shape}")

siglip_cache = CACHE_DIR / "siglip_embeddings.npy"
if siglip_cache.exists():
    siglip_embeds = np.load(siglip_cache)
    print(f"SigLIP2 cache hit: {siglip_embeds.shape}")
else:
    print("Encoding with SigLIP2...")
    siglip_embeds = encode_images(siglip_model, siglip_preprocess, sample_paths)
    np.save(siglip_cache, siglip_embeds)
    print(f"Saved SigLIP2 embeddings: {siglip_embeds.shape}")


Encoding with CLIP...
Saved CLIP embeddings: (500, 512)
Encoding with SigLIP2...
Saved SigLIP2 embeddings: (500, 768)


## 6. SAM2 Segmentation + Region Encoding (the slow part)

Each image is segmented by SAM2; each region is cropped (with 25% pad) and encoded by SigLIP2. Saves region embeddings, region-to-image map, bboxes, areas, and SAM2 confidences to one pickle.


In [7]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

print("Loading SAM2 Hiera-Base+...")
sam2 = build_sam2(SAM2_CFG, str(SAM2_CKPT), device=DEVICE)
mask_generator = SAM2AutomaticMaskGenerator(
    sam2, points_per_side=16, pred_iou_thresh=0.7,
    stability_score_thresh=0.85, min_mask_region_area=200,
)
print("SAM2 loaded.")

def crops_and_meta(image_pil, masks, pad_ratio=0.25, min_size=20):
    W, H = image_pil.size
    crops, areas, ious, bboxes = [], [], [], []
    for m in masks:
        x, y, w, h = m["bbox"]
        if w < min_size or h < min_size: continue
        pad_w, pad_h = w * pad_ratio, h * pad_ratio
        x0 = int(max(0, x - pad_w)); y0 = int(max(0, y - pad_h))
        x1 = int(min(W, x + w + pad_w)); y1 = int(min(H, y + h + pad_h))
        crops.append(image_pil.crop((x0, y0, x1, y1)))
        areas.append((x1 - x0) * (y1 - y0) / (W * H))
        ious.append(float(m.get("predicted_iou", 0.0)))
        bboxes.append((x0, y0, x1, y1))
    return crops, areas, ious, bboxes

def encode_pil_batch_siglip(pil_imgs, batch_size=32):
    if not pil_imgs:
        return np.zeros((0, siglip_embeds.shape[1]), dtype=np.float32)
    embs = []
    for i in range(0, len(pil_imgs), batch_size):
        x = torch.stack([siglip_preprocess(im) for im in pil_imgs[i:i+batch_size]]).to(DEVICE)
        with torch.no_grad():
            e = siglip_model.encode_image(x)
            e = e / e.norm(dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)


Loading SAM2 Hiera-Base+...
SAM2 loaded.


In [8]:
sam_cache = CACHE_DIR / "sam2_region_index.pkl"

if sam_cache.exists():
    with open(sam_cache, "rb") as f:
        region_index = pickle.load(f)
    print(f"SAM2 region index cache hit: {region_index['embeddings'].shape[0]} regions")
else:
    print("Running SAM2 + SigLIP2 region indexing...")
    region_embeds_list, region_to_img, region_areas, region_ious, region_bboxes = [], [], [], [], []
    for img_idx, p in enumerate(sample_paths):
        if img_idx % 20 == 0:
            print(f"  Image {img_idx}/{len(sample_paths)}")
        try:
            img_pil = Image.open(p).convert("RGB")
            masks = mask_generator.generate(np.array(img_pil))
            crops, areas, ious, bboxes = crops_and_meta(img_pil, masks)
            if crops:
                emb = encode_pil_batch_siglip(crops)
                region_embeds_list.append(emb)
                region_to_img.extend([img_idx] * len(emb))
                region_areas.extend(areas)
                region_ious.extend(ious)
                region_bboxes.extend(bboxes)
        except Exception as e:
            print(f"  Skipping image {img_idx}: {e}")

    region_index = {
        "embeddings": np.vstack(region_embeds_list) if region_embeds_list else np.zeros((0, siglip_embeds.shape[1])),
        "region_to_image": np.array(region_to_img, dtype=np.int32),
        "areas": np.array(region_areas, dtype=np.float32),
        "ious": np.array(region_ious, dtype=np.float32),
        "bboxes": np.array(region_bboxes, dtype=np.int32),
    }
    with open(sam_cache, "wb") as f:
        pickle.dump(region_index, f)
    print(f"Saved region index: {region_index['embeddings'].shape[0]} regions across {NUM_IMAGES} images")

print(f"Mean regions per image: {region_index['embeddings'].shape[0] / NUM_IMAGES:.1f}")


Running SAM2 + SigLIP2 region indexing...
  Image 0/500


/usr/local/lib/python3.12/dist-packages/sam2/sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (/usr/local/lib/python3.12/dist-packages/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(


  Image 20/500
  Image 40/500
  Image 60/500
  Image 80/500
  Image 100/500
  Image 120/500
  Image 140/500
  Image 160/500
  Image 180/500
  Image 200/500
  Image 220/500
  Image 240/500
  Image 260/500
  Image 280/500
  Image 300/500
  Image 320/500
  Image 340/500
  Image 360/500
  Image 380/500
  Image 400/500
  Image 420/500
  Image 440/500
  Image 460/500
  Image 480/500
Saved region index: 16361 regions across 500 images
Mean regions per image: 32.7


## 7. Verify and Summarize


In [9]:
print("Pipeline outputs (all saved to Drive):")
for name in ["sample_manifest.pkl", "clip_embeddings.npy", "siglip_embeddings.npy", "sam2_region_index.pkl"]:
    p = CACHE_DIR / name
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  OK  {name:<28} {size_mb:>7.2f} MB")
    else:
        print(f"  MISSING {name}")

print()
print("Next step: run 03_trained_scoring_head.ipynb to train models on these features.")


Pipeline outputs (all saved to Drive):
  OK  sample_manifest.pkl             0.03 MB
  OK  clip_embeddings.npy             1.02 MB
  OK  siglip_embeddings.npy           1.54 MB
  OK  sam2_region_index.pkl          50.72 MB

Next step: run 03_trained_scoring_head.ipynb to train models on these features.


## Summary

This notebook is the **dataset-generation layer**. It consumes raw COCO images and produces the four cached artifacts that downstream notebooks (training, explainability) consume.

After this completes successfully, you should never need to re-run it unless:
- You change `NUM_IMAGES`
- You change SAM2 hyperparameters (`points_per_side`, `pred_iou_thresh`, etc.)
- You switch image encoders

